In [1]:
import os
import zipfile
import json
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd

from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from torchvision import models, transforms

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
print(device := torch.device("cuda" if torch.cuda.is_available() else "cpu"))

cuda


In [3]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


unzip 

In [5]:
zip_path = "/content/drive/MyDrive/scenario23_dev_w_resources.zip"
print("Exists:", os.path.exists(zip_path))

Exists: True


In [7]:
extract_dir = "/content/dataset"

os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_dir)

print("Unzipped to:", extract_dir)

Unzipped to: /content/dataset


In [8]:
for root, dirs, files in os.walk(extract_dir):
    if "scenario23" in root.lower():
        print(root)

/content/dataset/scenario23_dev
/content/dataset/scenario23_dev/unit1
/content/dataset/scenario23_dev/unit1/camera_data
/content/dataset/scenario23_dev/unit1/mmWave_data
/content/dataset/scenario23_dev/unit1/GPS_data
/content/dataset/scenario23_dev/resources
/content/dataset/scenario23_dev/resources/bbox_labels_final
/content/dataset/scenario23_dev/unit2
/content/dataset/scenario23_dev/unit2/pitch
/content/dataset/scenario23_dev/unit2/roll
/content/dataset/scenario23_dev/unit2/distance
/content/dataset/scenario23_dev/unit2/x_speed
/content/dataset/scenario23_dev/unit2/GPS_data
/content/dataset/scenario23_dev/unit2/y_speed
/content/dataset/scenario23_dev/unit2/speed
/content/dataset/scenario23_dev/unit2/z_speed
/content/dataset/scenario23_dev/unit2/altitude
/content/dataset/scenario23_dev/unit2/height


In [9]:
! find /content/dataset -maxdepth 4 | head -200

/content/dataset
/content/dataset/scenario23_dev
/content/dataset/scenario23_dev/unit1
/content/dataset/scenario23_dev/unit1/camera_data
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_2011_17_04_02.jpg
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_10000_17_56_03.jpg
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_10001_17_56_03.jpg
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_10002_17_56_03.jpg
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_10003_17_56_03.jpg
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_10004_17_56_03.jpg
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_10005_17_56_03.jpg
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_10006_17_56_03.jpg
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_10007_17_56_04.jpg
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_10008_17_56_04.jpg
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_10009_17_56_04

In [10]:
DATA_ROOT = "/content/dataset/scenario23_dev"   # update if needed
print("Exists:", os.path.exists(DATA_ROOT))

Exists: True


In [11]:
csv_files = []
for root, dirs, files in os.walk(DATA_ROOT):
    for f in files:
        if f.lower().endswith(".csv"):
            csv_files.append(os.path.join(root, f))

print("Total CSV files:", len(csv_files))
for f in csv_files[:50]:
    print(f)

Total CSV files: 1
/content/dataset/scenario23_dev/scenario23.csv


In [12]:
image_exts = (".jpg", ".jpeg", ".png", ".bmp")
img_examples = []

for root, dirs, files in os.walk(DATA_ROOT):
    for f in files:
        if f.lower().endswith(image_exts):
            img_examples.append(os.path.join(root, f))

print("Total sample image hits:", len(img_examples))
for x in img_examples[:20]:
    print(x)

Total sample image hits: 11387
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_379_16_59_31.jpg
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_3857_17_09_15.jpg
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_2719_17_05_56.jpg
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_10364_17_56_56.jpg
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_5298_17_14_32.jpg
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_10949_17_58_46.jpg
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_832_17_00_43.jpg
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_6599_17_45_41.jpg
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_10597_17_57_49.jpg
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_3821_17_09_08.jpg
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_11224_17_59_27.jpg
/content/dataset/scenario23_dev/unit1/camera_data/image_BS1_9428_17_54_35.jpg
/content/dataset/scenario23_dev

In [13]:
for path in csv_files[:10]:
    print("="*100)
    print(path)
    try:
        df_tmp = pd.read_csv(path)
        print(df_tmp.head())
        print(df_tmp.columns.tolist())
        print(df_tmp.shape)
    except Exception as e:
        print("Could not read:", e)

/content/dataset/scenario23_dev/scenario23.csv
   index                                     unit1_rgb  \
0      1  ./unit1/camera_data/image_BS1_1_16_58_42.jpg   
1      2  ./unit1/camera_data/image_BS1_2_16_58_42.jpg   
2      3  ./unit1/camera_data/image_BS1_3_16_58_42.jpg   
3      4  ./unit1/camera_data/image_BS1_4_16_58_42.jpg   
4      5  ./unit1/camera_data/image_BS1_5_16_58_42.jpg   

                          unit1_pwr_60ghz                          unit1_loc  \
0  ./unit1/mmWave_data/mmWave_power_1.txt  ./unit1/GPS_data/gps_location.txt   
1  ./unit1/mmWave_data/mmWave_power_2.txt  ./unit1/GPS_data/gps_location.txt   
2  ./unit1/mmWave_data/mmWave_power_3.txt  ./unit1/GPS_data/gps_location.txt   
3  ./unit1/mmWave_data/mmWave_power_4.txt  ./unit1/GPS_data/gps_location.txt   
4  ./unit1/mmWave_data/mmWave_power_5.txt  ./unit1/GPS_data/gps_location.txt   

                             unit2_loc                unit2_speed  \
0  ./unit2/GPS_data/gps_location_1.txt  ./unit2/speed/

In [14]:
candidate_tables = []

for path in csv_files:
    try:
        df_tmp = pd.read_csv(path, nrows=5)
        cols = [c.lower() for c in df_tmp.columns]
        if any("beam" in c for c in cols) or any("rgb" in c for c in cols):
            candidate_tables.append(path)
    except:
        pass

print("Candidate metadata tables:")
for p in candidate_tables:
    print(p)

Candidate metadata tables:
/content/dataset/scenario23_dev/scenario23.csv


inspect important columns 

# for loop diye column dekhe column mapping korbo 

In [15]:
meta_csv = candidate_tables[0]   # update manually if needed
df = pd.read_csv(meta_csv)
print(df.shape)
print(df.head())
print(df.columns.tolist())

(11387, 17)
   index                                     unit1_rgb  \
0      1  ./unit1/camera_data/image_BS1_1_16_58_42.jpg   
1      2  ./unit1/camera_data/image_BS1_2_16_58_42.jpg   
2      3  ./unit1/camera_data/image_BS1_3_16_58_42.jpg   
3      4  ./unit1/camera_data/image_BS1_4_16_58_42.jpg   
4      5  ./unit1/camera_data/image_BS1_5_16_58_42.jpg   

                          unit1_pwr_60ghz                          unit1_loc  \
0  ./unit1/mmWave_data/mmWave_power_1.txt  ./unit1/GPS_data/gps_location.txt   
1  ./unit1/mmWave_data/mmWave_power_2.txt  ./unit1/GPS_data/gps_location.txt   
2  ./unit1/mmWave_data/mmWave_power_3.txt  ./unit1/GPS_data/gps_location.txt   
3  ./unit1/mmWave_data/mmWave_power_4.txt  ./unit1/GPS_data/gps_location.txt   
4  ./unit1/mmWave_data/mmWave_power_5.txt  ./unit1/GPS_data/gps_location.txt   

                             unit2_loc                unit2_speed  \
0  ./unit2/GPS_data/gps_location_1.txt  ./unit2/speed/speed_1.txt   
1  ./unit2/GPS_data/

In [16]:
for col in df.columns:
    print(col)

index
unit1_rgb
unit1_pwr_60ghz
unit1_loc
unit2_loc
unit2_speed
unit2_altitude
unit2_distance
unit2_height
unit2_x-speed
unit2_y-speed
unit2_z-speed
unit2_pitch
unit2_roll
seq_index
time_stamp[UTC]
unit1_beam_index


column mapping 

In [17]:
COLUMN_MAP = {
    "image_col": None,
    "lat_col": None,
    "lon_col": None,
    "height_col": None,
    "distance_col": None,
    "beam32_col": None,
    "beam64_col": None,
}

In [19]:
cols = df.columns.tolist()
lower_map = {c.lower(): c for c in cols}

print("Possible image columns:")
for c in cols:
    if "rgb" in c.lower() or "image" in c.lower() or "camera" in c.lower():
        print(c)

print("\nPossible latitude/longitude columns:")
for c in cols:
    if "lat" in c.lower() or "lon" in c.lower() or "long" in c.lower() or "gps" in c.lower():
        print(c)

print("\nPossible height/distance columns:")
for c in cols:
    if "height" in c.lower() or "dist" in c.lower():
        print(c)

print("\nPossible beam columns:")
for c in cols:
    if "beam" in c.lower():
        print(c)

Possible image columns:
unit1_rgb

Possible latitude/longitude columns:

Possible height/distance columns:
unit2_distance
unit2_height

Possible beam columns:
unit1_beam_index


In [20]:
COLUMN_MAP = {
    "image_col": "unit1_rgb",
    "lat_col": "unit2_loc_0",      # update
    "lon_col": "unit2_loc_1",      # update
    "height_col": "unit2_height",  # update
    "distance_col": "unit1_unit2_distance",  # update
    "beam32_col": "unit1_beam_32",
    "beam64_col": "unit1_beam_64",
}

COLUMN_MAP

{'image_col': 'unit1_rgb',
 'lat_col': 'unit2_loc_0',
 'lon_col': 'unit2_loc_1',
 'height_col': 'unit2_height',
 'distance_col': 'unit1_unit2_distance',
 'beam32_col': 'unit1_beam_32',
 'beam64_col': 'unit1_beam_64'}

In [21]:
for key, col in COLUMN_MAP.items():
    if col is not None and col in df.columns:
        print(f"{key} -> {col} | nulls = {df[col].isna().sum()}")

image_col -> unit1_rgb | nulls = 0
height_col -> unit2_height | nulls = 0


In [22]:
def make_abs_path(p):
    if pd.isna(p):
        return None
    p = str(p)
    if os.path.isabs(p):
        return p
    return os.path.join("/content", p.lstrip("/"))

df["image_path"] = df[COLUMN_MAP["image_col"]].apply(make_abs_path)

print(df["image_path"].head())
print("Existing images:", df["image_path"].apply(lambda x: os.path.exists(x) if isinstance(x, str) else False).sum())

0    /content/./unit1/camera_data/image_BS1_1_16_58...
1    /content/./unit1/camera_data/image_BS1_2_16_58...
2    /content/./unit1/camera_data/image_BS1_3_16_58...
3    /content/./unit1/camera_data/image_BS1_4_16_58...
4    /content/./unit1/camera_data/image_BS1_5_16_58...
Name: image_path, dtype: object
Existing images: 0


beam index ta ber korlam 

In [23]:
label_candidates = [
    COLUMN_MAP.get("beam32_col"),
    COLUMN_MAP.get("beam64_col"),
]
label_candidates += [
    c for c in df.columns
    if "beam" in c.lower() and ("index" in c.lower() or "label" in c.lower())
]

label_col = next((c for c in label_candidates if c in df.columns), None)

if label_col is None:
    beam_cols = [c for c in df.columns if "beam" in c.lower()]
    raise ValueError(f"No usable beam label column found. Beam-related columns: {beam_cols}")

df["label"] = pd.to_numeric(df[label_col], errors="coerce")

if df["label"].isna().all():
    raise ValueError(f"Column '{label_col}' could not be converted to numeric labels.")

print("Using label column:", label_col)
print(df["label"].head())
print("Num classes:", df["label"].nunique())
print("Min label:", df["label"].min(), "Max label:", df["label"].max())

Using label column: unit1_beam_index
0    43
1    43
2    43
3    43
4    39
Name: label, dtype: int64
Num classes: 56
Min label: 2 Max label: 60


In [24]:
needed_cols = ["label", "image_path"]
feature_cols = []

for k in ["lat_col", "lon_col", "height_col", "distance_col"]:
    col = COLUMN_MAP[k]
    if col is not None and col in df.columns:
        feature_cols.append(col)

needed_cols += feature_cols

work_df = df.copy()

for col in feature_cols + ["label"]:
    work_df[col] = pd.to_numeric(work_df[col], errors="coerce")

work_df = work_df.dropna(subset=["label"])
work_df["label"] = work_df["label"].astype(int)

print(work_df.shape)
work_df.head()

(11387, 19)


,index,unit1_rgb,unit1_pwr_60ghz,unit1_loc,unit2_loc,unit2_speed,unit2_altitude,unit2_distance,unit2_height,unit2_x-speed,unit2_y-speed,unit2_z-speed,unit2_pitch,unit2_roll,seq_index,time_stamp[UTC],unit1_beam_index,image_path,label
0,1,./unit1/camera_data/image_BS1_1_16_58_42.jpg,./unit1/mmWave_data/mmWave_power_1.txt,./unit1/GPS_data/gps_location.txt,./unit2/GPS_data/gps_location_1.txt,./unit2/speed/speed_1.txt,./unit2/altitude/altitude_1.txt,./unit2/distance/distance_1.txt,NaN,./unit2/x_speed/x_speed_1.txt,./unit2/y_speed/y_speed_1.txt,./unit2/z_speed/z_speed_1.txt,./unit2/pitch/pitch_1.txt,./unit2/roll/roll_1.txt,1,['16-58-42-0'],43,/content/./unit1/camera_data/image_BS1_1_16_58...,43
1,2,./unit1/camera_data/image_BS1_2_16_58_42.jpg,./unit1/mmWave_data/mmWave_power_2.txt,./unit1/GPS_data/gps_location.txt,./unit2/GPS_data/gps_location_2.txt,./unit2/speed/speed_2.txt,./unit2/altitude/altitude_2.txt,./unit2/distance/distance_2.txt,NaN,./unit2/x_speed/x_speed_2.txt,./unit2/y_speed/y_speed_2.txt,./unit2/z_speed/z_speed_2.txt,./unit2/pitch/pitch_2.txt,./unit2/roll/roll_2.txt,1,['16-58-42-142'],43,/content/./unit1/camera_data/image_BS1_2_16_58...,43
2,3,./unit1/camera_data/image_BS1_3_16_58_42.jpg,./unit1/mmWave_data/mmWave_power_3.txt,./unit1/GPS_data/gps_location.txt,./unit2/GPS_data/gps_location_3.txt,./unit2/speed/speed_3.txt,./unit2/altitude/altitude_3.txt,./unit2/distance/distance_3.txt,NaN,./unit2/x_speed/x_speed_3.txt,./unit2/y_speed/y_speed_3.txt,./unit2/z_speed/z_speed_3.txt,./unit2/pitch/pitch_3.txt,./unit2/roll/roll_3.txt,1,['16-58-42-284'],43,/content/./unit1/camera_data/image_BS1_3_16_58...,43
3,4,./unit1/camera_data/image_BS1_4_16_58_42.jpg,./unit1/mmWave_data/mmWave_power_4.txt,./unit1/GPS_data/gps_location.txt,./unit2/GPS_data/gps_location_4.txt,./unit2/speed/speed_4.txt,./unit2/altitude/altitude_4.txt,./unit2/distance/distance_4.txt,NaN,./unit2/x_speed/x_speed_4.txt,./unit2/y_speed/y_speed_4.txt,./unit2/z_speed/z_speed_4.txt,./unit2/pitch/pitch_4.txt,./unit2/roll/roll_4.txt,1,['16-58-42-426'],43,/content/./unit1/camera_data/image_BS1_4_16_58...,43
4,5,./unit1/camera_data/image_BS1_5_16_58_42.jpg,./unit1/mmWave_data/mmWave_power_5.txt,./unit1/GPS_data/gps_location.txt,./unit2/GPS_data/gps_location_5.txt,./unit2/speed/speed_5.txt,./unit2/altitude/altitude_5.txt,./unit2/distance/distance_5.txt,NaN,./unit2/x_speed/x_speed_5.txt,./unit2/y_speed/y_speed_5.txt,./unit2/z_speed/z_speed_5.txt,./unit2/pitch/pitch_5.txt,./unit2/roll/roll_5.txt,1,['16-58-42-568'],39,/content/./unit1/camera_data/image_BS1_5_16_58...,39


In [25]:
print(sorted(work_df["label"].unique())[:10], "...", sorted(work_df["label"].unique())[-10:])

[np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11)] ... [np.int64(51), np.int64(52), np.int64(53), np.int64(54), np.int64(55), np.int64(56), np.int64(57), np.int64(58), np.int64(59), np.int64(60)]


In [26]:
unique_labels = sorted(work_df["label"].unique())

if min(unique_labels) == 1 and max(unique_labels) == 32:
    work_df["label"] = work_df["label"] - 1
elif min(unique_labels) == 0 and max(unique_labels) == 31:
    pass
else:
    print("Unexpected label range:", min(unique_labels), max(unique_labels))

print("Final label range:", work_df["label"].min(), work_df["label"].max())

Unexpected label range: 2 60
Final label range: 2 60


In [27]:
position_cols = [COLUMN_MAP["lat_col"], COLUMN_MAP["lon_col"]]
position_height_cols = [COLUMN_MAP["lat_col"], COLUMN_MAP["lon_col"], COLUMN_MAP["height_col"]]
position_height_distance_cols = [COLUMN_MAP["lat_col"], COLUMN_MAP["lon_col"], COLUMN_MAP["height_col"], COLUMN_MAP["distance_col"]]

print(position_cols)
print(position_height_cols)
print(position_height_distance_cols)

['unit2_loc_0', 'unit2_loc_1']
['unit2_loc_0', 'unit2_loc_1', 'unit2_height']
['unit2_loc_0', 'unit2_loc_1', 'unit2_height', 'unit1_unit2_distance']


In [28]:
import re

CSV_DIR = os.path.dirname(meta_csv) if isinstance(globals().get("meta_csv"), str) else "/content"
_FLOAT_RE = re.compile(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?")

def resolve_ref_path(ref):
    if pd.isna(ref):
        return None
    ref = str(ref).strip()
    if not ref:
        return None
    return ref if os.path.isabs(ref) else os.path.normpath(os.path.join(CSV_DIR, ref))

def read_numbers_from_ref(ref):
    path = resolve_ref_path(ref)
    if not path or not os.path.exists(path):
        return []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return [float(x) for x in _FLOAT_RE.findall(f.read())]

def read_scalar_from_ref(ref):
    values = read_numbers_from_ref(ref)
    return values[0] if values else np.nan

def read_gps_from_ref(ref):
    values = read_numbers_from_ref(ref)
    if len(values) >= 2:
        return pd.Series({"lat": values[0], "lon": values[1]})
    return pd.Series({"lat": np.nan, "lon": np.nan})

raw_df = df if "df" in globals() else work_df

def pick_ref_source(candidates):
    for c in candidates:
        if c in raw_df.columns and raw_df[c].notna().any():
            return c
    return next((c for c in candidates if c in raw_df.columns), None)

if COLUMN_MAP.get("lat_col") not in work_df.columns or COLUMN_MAP.get("lon_col") not in work_df.columns:
    loc_src = pick_ref_source(["unit2_loc", "unit1_loc"])
    if loc_src is not None:
        gps_values = raw_df[loc_src].apply(read_gps_from_ref)
        work_df["unit2_lat"] = gps_values["lat"]
        work_df["unit2_lon"] = gps_values["lon"]
        COLUMN_MAP["lat_col"] = "unit2_lat"
        COLUMN_MAP["lon_col"] = "unit2_lon"

if COLUMN_MAP.get("height_col") not in work_df.columns or work_df[COLUMN_MAP.get("height_col")].notna().sum() == 0:
    height_src = pick_ref_source(["unit2_height", "unit2_altitude"])
    if height_src is not None:
        work_df["unit2_height_value"] = raw_df[height_src].apply(read_scalar_from_ref)
        COLUMN_MAP["height_col"] = "unit2_height_value"

if COLUMN_MAP.get("distance_col") not in work_df.columns or work_df[COLUMN_MAP.get("distance_col")].notna().sum() == 0:
    distance_src = pick_ref_source(["unit2_distance", "unit1_unit2_distance"])
    if distance_src is not None:
        work_df["unit2_distance_value"] = raw_df[distance_src].apply(read_scalar_from_ref)
        COLUMN_MAP["distance_col"] = "unit2_distance_value"

resolved_cols = [
    COLUMN_MAP.get("lat_col"),
    COLUMN_MAP.get("lon_col"),
    COLUMN_MAP.get("height_col"),
    COLUMN_MAP.get("distance_col"),
]
for col in resolved_cols:
    if col in work_df.columns:
        work_df[col] = pd.to_numeric(work_df[col], errors="coerce")

position_cols = [c for c in [COLUMN_MAP.get("lat_col"), COLUMN_MAP.get("lon_col")] if c in work_df.columns]
position_height_cols = position_cols + ([COLUMN_MAP["height_col"]] if COLUMN_MAP.get("height_col") in work_df.columns else [])
position_height_distance_cols = position_height_cols + ([COLUMN_MAP["distance_col"]] if COLUMN_MAP.get("distance_col") in work_df.columns else [])

df_pos = work_df.dropna(subset=position_cols).copy() if len(position_cols) == 2 else work_df.iloc[0:0].copy()
df_pos_h = work_df.dropna(subset=position_height_cols).copy() if len(position_height_cols) == 3 else work_df.iloc[0:0].copy()
df_pos_h_d = work_df.dropna(subset=position_height_distance_cols).copy() if len(position_height_distance_cols) == 4 else work_df.iloc[0:0].copy()
df_img = work_df[work_df["image_path"].apply(lambda x: os.path.exists(x) if isinstance(x, str) else False)].copy()

print("Resolved position cols:", position_cols)
print("Resolved position + height cols:", position_height_cols)
print("Resolved position + height + distance cols:", position_height_distance_cols)
print("Position only:", df_pos.shape)
print("Position + Height:", df_pos_h.shape)
print("Position + Height + Distance:", df_pos_h_d.shape)
print("Image:", df_img.shape)

Resolved position cols: ['unit2_lat', 'unit2_lon']
Resolved position + height cols: ['unit2_lat', 'unit2_lon', 'unit2_height_value']
Resolved position + height + distance cols: ['unit2_lat', 'unit2_lon', 'unit2_height_value', 'unit2_distance_value']
Position only: (11387, 23)
Position + Height: (11387, 23)
Position + Height + Distance: (11387, 23)
Image: (0, 23)


In [29]:
def split_df(dataframe, test_size=0.30, seed=42):
    # remove classes with fewer than 2 samples
    label_counts = dataframe["label"].value_counts()
    valid_labels = label_counts[label_counts >= 2].index

    filtered_df = dataframe[dataframe["label"].isin(valid_labels)].copy()

    print("Original shape:", dataframe.shape)
    print("Filtered shape:", filtered_df.shape)
    print("Removed rows:", len(dataframe) - len(filtered_df))

    train_df, test_df = train_test_split(
        filtered_df,
        test_size=test_size,
        random_state=seed,
        stratify=filtered_df["label"]
    )

    return train_df.reset_index(drop=True), test_df.reset_index(drop=True)

In [30]:
df_pos = work_df.dropna(subset=position_cols).copy()
df_pos_h = work_df.dropna(subset=position_height_cols).copy()
df_pos_h_d = work_df.dropna(subset=position_height_distance_cols).copy()
df_img = work_df[work_df["image_path"].apply(lambda x: os.path.exists(x) if isinstance(x, str) else False)].copy()

print("Position only:", df_pos.shape)
print("Position + Height:", df_pos_h.shape)
print("Position + Height + Distance:", df_pos_h_d.shape)
print("Image:", df_img.shape)

Position only: (11387, 23)
Position + Height: (11387, 23)
Position + Height + Distance: (11387, 23)
Image: (0, 23)


In [32]:
def split_df(dataframe, test_size=0.30, seed=42):
    train_df, test_df = train_test_split(
        dataframe,
        test_size=test_size,
        random_state=seed,
        stratify=dataframe["label"]
    )
    return train_df.reset_index(drop=True), test_df.reset_index(drop=True)

train_pos, test_pos = split_df(df_pos)
train_pos_h, test_pos_h = split_df(df_pos_h)
train_pos_h_d, test_pos_h_d = split_df(df_pos_h_d)
train_img, test_img = split_df(df_img)

print(train_pos.shape, test_pos.shape)
print(train_pos_h.shape, test_pos_h.shape)
print(train_pos_h_d.shape, test_pos_h_d.shape)
print(train_img.shape, test_img.shape)

ValueError: The least populated class in y has only 1 member, which is too few. The minimum number of groups for any class cannot be less than 2.

In [33]:
print("df_pos shape:", df_pos.shape)
print("label min/max:", df_pos["label"].min(), df_pos["label"].max())
print(df_pos["label"].value_counts().sort_values().head(20))

df_pos shape: (11387, 23)
label min/max: 2 60
label
12     1
60     1
35     1
56     1
24     1
53     2
5      3
15     6
57     7
10     8
9      9
58     9
25    10
54    14
8     25
16    32
59    34
52    35
45    36
46    37
Name: count, dtype: int64


In [34]:
from sklearn.model_selection import train_test_split

def split_df_debug(dataframe, test_size=0.30, seed=42, label_col="label", use_stratify=False):
    df_local = dataframe.copy()
    empty_like = dataframe.iloc[0:0].copy()

    if label_col not in df_local.columns:
        print("Before split:", df_local.shape)
        print(f"Missing label column: {label_col}")
        return empty_like.reset_index(drop=True), empty_like.reset_index(drop=True)

    # keep numeric labels only
    df_local[label_col] = pd.to_numeric(df_local[label_col], errors="coerce")
    df_local = df_local.dropna(subset=[label_col]).copy()
    if not df_local.empty:
        df_local[label_col] = df_local[label_col].astype(int)

    print("Before split:", df_local.shape)
    print("Unique labels:", df_local[label_col].nunique())
    print("Smallest class counts:")
    print(df_local[label_col].value_counts().sort_values().head(10))

    if df_local.empty:
        print("Dataset is empty after label cleanup; returning empty train/test splits.")
        return empty_like.reset_index(drop=True), empty_like.reset_index(drop=True)

    if use_stratify:
        label_counts = df_local[label_col].value_counts()
        valid_labels = label_counts[label_counts >= 2].index
        if len(valid_labels) < label_counts.size:
            removed = label_counts[label_counts < 2]
            print("Dropping classes with <2 samples for stratified split:")
            print(removed)
            df_local = df_local[df_local[label_col].isin(valid_labels)].copy()

        if df_local.empty or df_local[label_col].nunique() < 2:
            print("Not enough data for a stratified split; falling back to non-stratified split.")
            stratify_arg = None
        else:
            stratify_arg = df_local[label_col]
    else:
        stratify_arg = None

    if len(df_local) < 2:
        print("Dataset has fewer than 2 rows; returning all rows as train and an empty test split.")
        return df_local.reset_index(drop=True), empty_like.reset_index(drop=True)

    try:
        train_df, test_df = train_test_split(
            df_local,
            test_size=test_size,
            random_state=seed,
            shuffle=True,
            stratify=stratify_arg
        )
    except ValueError as e:
        print(f"Split skipped: {e}")
        return df_local.reset_index(drop=True), empty_like.reset_index(drop=True)

    return train_df.reset_index(drop=True), test_df.reset_index(drop=True)

In [35]:
def run_split_or_empty(name, dataframe, **kwargs):
    empty_like = dataframe.iloc[0:0].copy()

    if dataframe.empty:
        print(f"{name}: dataset is empty; returning empty train/test splits.")
        return empty_like, empty_like.copy()

    if "label" not in dataframe.columns:
        print(f"{name}: missing 'label' column; returning empty train/test splits.")
        return empty_like, empty_like.copy()

    try:
        return split_df_debug(dataframe, **kwargs)
    except ValueError as e:
        print(f"{name}: split skipped -> {e}")
        return dataframe.reset_index(drop=True), empty_like.copy()

train_pos, test_pos = run_split_or_empty("df_pos", df_pos, use_stratify=False)
train_pos_h, test_pos_h = run_split_or_empty("df_pos_h", df_pos_h, use_stratify=False)
train_pos_h_d, test_pos_h_d = run_split_or_empty("df_pos_h_d", df_pos_h_d, use_stratify=False)
train_img, test_img = run_split_or_empty("df_img", df_img, use_stratify=False)

print(train_pos.shape, test_pos.shape)
print(train_pos_h.shape, test_pos_h.shape)
print(train_pos_h_d.shape, test_pos_h_d.shape)
print(train_img.shape, test_img.shape)

Before split: (11387, 23)
Unique labels: 56
Smallest class counts:
label
12    1
60    1
35    1
56    1
24    1
53    2
5     3
15    6
57    7
10    8
Name: count, dtype: int64
Before split: (11387, 23)
Unique labels: 56
Smallest class counts:
label
12    1
60    1
35    1
56    1
24    1
53    2
5     3
15    6
57    7
10    8
Name: count, dtype: int64
Before split: (11387, 23)
Unique labels: 56
Smallest class counts:
label
12    1
60    1
35    1
56    1
24    1
53    2
5     3
15    6
57    7
10    8
Name: count, dtype: int64
df_img: dataset is empty; returning empty train/test splits.
(7970, 23) (3417, 23)
(7970, 23) (3417, 23)
(7970, 23) (3417, 23)
(0, 23) (0, 23)


In [36]:
print("Paper target ~ train 8402 / test 3602")
print("Our splits:")
print("POS:", len(train_pos), len(test_pos))
print("POS+H:", len(train_pos_h), len(test_pos_h))
print("POS+H+D:", len(train_pos_h_d), len(test_pos_h_d))
print("IMG:", len(train_img), len(test_img))

Paper target ~ train 8402 / test 3602
Our splits:
POS: 7970 3417
POS+H: 7970 3417
POS+H+D: 7970 3417
IMG: 0 0


In [37]:
save_dir = "/content/processed_scenario23"
os.makedirs(save_dir, exist_ok=True)

train_pos.to_csv(f"{save_dir}/train_pos.csv", index=False)
test_pos.to_csv(f"{save_dir}/test_pos.csv", index=False)

train_pos_h.to_csv(f"{save_dir}/train_pos_h.csv", index=False)
test_pos_h.to_csv(f"{save_dir}/test_pos_h.csv", index=False)

train_pos_h_d.to_csv(f"{save_dir}/train_pos_h_d.csv", index=False)
test_pos_h_d.to_csv(f"{save_dir}/test_pos_h_d.csv", index=False)

train_img.to_csv(f"{save_dir}/train_img.csv", index=False)
test_img.to_csv(f"{save_dir}/test_img.csv", index=False)

print("Saved to:", save_dir)

Saved to: /content/processed_scenario23


In [38]:
print(df.columns.tolist())
print(candidate_tables)
for i, p in enumerate(candidate_tables):
    print(i, p)

['index', 'unit1_rgb', 'unit1_pwr_60ghz', 'unit1_loc', 'unit2_loc', 'unit2_speed', 'unit2_altitude', 'unit2_distance', 'unit2_height', 'unit2_x-speed', 'unit2_y-speed', 'unit2_z-speed', 'unit2_pitch', 'unit2_roll', 'seq_index', 'time_stamp[UTC]', 'unit1_beam_index', 'image_path', 'label']
['/content/dataset/scenario23_dev/scenario23.csv']
0 /content/dataset/scenario23_dev/scenario23.csv


In [39]:
print("POS:", len(train_pos), len(test_pos))
print("POS+H:", len(train_pos_h), len(test_pos_h))
print("POS+H+D:", len(train_pos_h_d), len(test_pos_h_d))
print("IMG:", len(train_img), len(test_img))

POS: 7970 3417
POS+H: 7970 3417
POS+H+D: 7970 3417
IMG: 0 0


In [40]:
print("Label min:", work_df["label"].min())
print("Label max:", work_df["label"].max())
print("Unique labels:", sorted(work_df["label"].unique())[:10], "...", sorted(work_df["label"].unique())[-10:])
print(work_df["label"].value_counts().sort_values().head(20))

Label min: 2
Label max: 60
Unique labels: [np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11)] ... [np.int64(51), np.int64(52), np.int64(53), np.int64(54), np.int64(55), np.int64(56), np.int64(57), np.int64(58), np.int64(59), np.int64(60)]
label
12     1
60     1
35     1
56     1
24     1
53     2
5      3
15     6
57     7
10     8
9      9
58     9
25    10
54    14
8     25
16    32
59    34
52    35
45    36
46    37
Name: count, dtype: int64


In [41]:
print(work_df["label"].value_counts().sort_values().head(20))

label
12     1
60     1
35     1
56     1
24     1
53     2
5      3
15     6
57     7
10     8
9      9
58     9
25    10
54    14
8     25
16    32
59    34
52    35
45    36
46    37
Name: count, dtype: int64


In [42]:
print(df["image_path"].apply(lambda x: os.path.exists(x) if isinstance(x, str) else False).sum())
print("Total rows:", len(df))

0
Total rows: 11387


In [43]:
for c in df.columns:
    print(c)

index
unit1_rgb
unit1_pwr_60ghz
unit1_loc
unit2_loc
unit2_speed
unit2_altitude
unit2_distance
unit2_height
unit2_x-speed
unit2_y-speed
unit2_z-speed
unit2_pitch
unit2_roll
seq_index
time_stamp[UTC]
unit1_beam_index
image_path
label


In [44]:
print(df.columns.tolist())
print(candidate_tables)
print(work_df["label"].value_counts().sort_values().head(20))
df.head().T

['index', 'unit1_rgb', 'unit1_pwr_60ghz', 'unit1_loc', 'unit2_loc', 'unit2_speed', 'unit2_altitude', 'unit2_distance', 'unit2_height', 'unit2_x-speed', 'unit2_y-speed', 'unit2_z-speed', 'unit2_pitch', 'unit2_roll', 'seq_index', 'time_stamp[UTC]', 'unit1_beam_index', 'image_path', 'label']
['/content/dataset/scenario23_dev/scenario23.csv']
label
12     1
60     1
35     1
56     1
24     1
53     2
5      3
15     6
57     7
10     8
9      9
58     9
25    10
54    14
8     25
16    32
59    34
52    35
45    36
46    37
Name: count, dtype: int64


,0,1,2,3,4
index,1,2,3,4,5
unit1_rgb,./unit1/camera_data/image_BS1_1_16_58_42.jpg,./unit1/camera_data/image_BS1_2_16_58_42.jpg,./unit1/camera_data/image_BS1_3_16_58_42.jpg,./unit1/camera_data/image_BS1_4_16_58_42.jpg,./unit1/camera_data/image_BS1_5_16_58_42.jpg
unit1_pwr_60ghz,./unit1/mmWave_data/mmWave_power_1.txt,./unit1/mmWave_data/mmWave_power_2.txt,./unit1/mmWave_data/mmWave_power_3.txt,./unit1/mmWave_data/mmWave_power_4.txt,./unit1/mmWave_data/mmWave_power_5.txt
unit1_loc,./unit1/GPS_data/gps_location.txt,./unit1/GPS_data/gps_location.txt,./unit1/GPS_data/gps_location.txt,./unit1/GPS_data/gps_location.txt,./unit1/GPS_data/gps_location.txt
unit2_loc,./unit2/GPS_data/gps_location_1.txt,./unit2/GPS_data/gps_location_2.txt,./unit2/GPS_data/gps_location_3.txt,./unit2/GPS_data/gps_location_4.txt,./unit2/GPS_data/gps_location_5.txt
unit2_speed,./unit2/speed/speed_1.txt,./unit2/speed/speed_2.txt,./unit2/speed/speed_3.txt,./unit2/speed/speed_4.txt,./unit2/speed/speed_5.txt
unit2_altitude,./unit2/altitude/altitude_1.txt,./unit2/altitude/altitude_2.txt,./unit2/altitude/altitude_3.txt,./unit2/altitude/altitude_4.txt,./unit2/altitude/altitude_5.txt
unit2_distance,./unit2/distance/distance_1.txt,./unit2/distance/distance_2.txt,./unit2/distance/distance_3.txt,./unit2/distance/distance_4.txt,./unit2/distance/distance_5.txt
unit2_height,./unit2/height/height_1.txt,./unit2/height/height_2.txt,./unit2/height/height_3.txt,./unit2/height/height_4.txt,./unit2/height/height_5.txt
unit2_x-speed,./unit2/x_speed/x_speed_1.txt,./unit2/x_speed/x_speed_2.txt,./unit2/x_speed/x_speed_3.txt,./unit2/x_speed/x_speed_4.txt,./unit2/x_speed/x_speed_5.txt
